In [23]:
import pandas as pd
import re
from collections import Counter

In [24]:
texts = pd.read_csv("../data/raw/question_texts_texts.csv")

texts = texts[["sanity_text_id", "serialNumber", "title", "body"]].copy()

texts["title"] = texts["title"].fillna("").astype(str)
texts["body"] = texts["body"].fillna("").astype(str)

texts.head()

,sanity_text_id,serialNumber,title,body
0,01ff8ad2-ef5b-4cf2-8f1d-b44663790fa4,HT_105_+ÿrner,+ÿrner,## +ÿrner\r\n\r\n![+ÿrn med fisk i kl++rne. Foto....
1,0202b907-4516-4beb-8e07-759e473b5ea2,HT_43_Astronaut,Astronaut,## Astronaut\r\n\r\n![Astronaut viser tommel o...
2,02df17fa-3bfa-4f87-8cd5-d523c12c9a1f,HT_04_Snork,Snork,## Snork\r\n\r\nGÇô Jeg vil ikke sove p+Ñ rommet ...
3,035af603-c895-4f40-a3e8-4b05a07a09bd,HT_125_Miniatyrer-William,Miniatyrer - gjennom n+Ñl++yet,## Miniatyrer - gjennom n+Ñl++yet\r\n\r\n![Forsi...
4,05c18c0e-b6b5-40b8-9009-130b044693c6,HT_135_NRK,NRK,## NRK\r\n\r\n![NRK-bygget Marienlyst i Oslo](...


In [25]:
# Create quick length statistics to understand the text size distribution
texts["body_char_count"] = texts["body"].str.len()
texts["body_line_count"] = texts["body"].apply(lambda x: len(x.splitlines()))

texts[["body_char_count", "body_line_count"]].describe()

,body_char_count,body_line_count
count,159.000000,159.000000
mean,2050.226415,28.572327
std,985.597826,17.547930
min,159.000000,5.000000
25%,1297.000000,15.500000
50%,2140.000000,24.000000
75%,2686.500000,38.500000
max,5492.000000,107.000000


In [26]:
# Join all body text into one big string
all_body_text = " ".join(texts["body"].tolist())

# Count every character
char_counts = Counter(all_body_text)

# Keep only non-alphanumeric, non-space characters
special_chars = pd.DataFrame(
    [(char, count) for char, count in char_counts.items()
     if not char.isalnum() and not char.isspace()],
    columns=["char", "count"]
).sort_values("count", ascending=False)

# Show the most common special characters
special_chars.head(50)

,char,count
3,.,4675
7,",",1951
0,#,894
1,!,483
6,),414
5,(,413
4,],347
2,[,347
12,GÇô,244
9,-,222


In [6]:
# Many structural patterns happen at the beginning of lines
# For example: ## heading, - bullet, ![image], etc.

line_starts = Counter()

for text in texts["body"]:
    for line in text.splitlines():
        line = line.strip()
        if line:
            # Count the first 12 characters of each non-empty line
            line_starts[line[:12]] += 1

line_start_df = pd.DataFrame(
    line_starts.most_common(50),
    columns=["line_start", "count"]
)

line_start_df.head(50)

,line_start,count
0,###,16
1,### Levevis,14
2,### Beskrive,14
3,### Formerin,11
4,![N+ªrbilde a,11
5,Fra Nemi-bla,9
6,### Historie,8
7,Det er mange,8
8,![Illustrasj,7
9,Jeg synes de,6


In [47]:
# These are broad, known pattern families we want to search for.

pattern_checks = {
    # Markdown / markup related
    "markdown_heading": r"(?m)^\s*#{1,6}\s+",
    "markdown_image": r"!\[.*?\]\(.*?\)",
    "markdown_link": r"\[.*?\]\(.*?\)",
    "bullet_list": r"(?m)^\s*[-*]\s+",
    "numbered_list": r"(?m)^\s*\d+\.\s+",
    "html_tag": r"<[^>]+>",

    # Norwegian-specific quotation/dialogue related
    "guillemets_quotes": r"[-½-+]",
    "curly_double_quotes": r"[GÇ£GÇ¥]",
    "dialogue_dash": r"(?m)^\s*[GÇôGÇö-]\s+",
}

# Show the pattern names
list(pattern_checks.keys())

['markdown_heading',
 'markdown_image',
 'markdown_link',
 'bullet_list',
 'numbered_list',
 'html_tag',
 'guillemets_quotes',
 'curly_double_quotes',
 'dialogue_dash']

In [29]:
# Count how many rows contain each pattern at least once
pattern_summary = []

for name, pattern in pattern_checks.items():
    match_count = texts["body"].str.contains(pattern, regex=True, na=False).sum()
    pattern_summary.append((name, match_count))

pattern_summary_df = pd.DataFrame(pattern_summary, columns=["pattern_name", "row_count"])
pattern_summary_df = pattern_summary_df.sort_values("row_count", ascending=False)

pattern_summary_df

,pattern_name,row_count
0,markdown_heading,159
1,markdown_image,157
2,markdown_link,157
6,guillemets_quotes,46
8,dialogue_dash,13
7,curly_double_quotes,4
5,html_tag,2
4,numbered_list,1
3,bullet_list,1


In [30]:
# Count the total number of matches across the entire dataset
total_match_summary = []

for name, pattern in pattern_checks.items():
    total_matches = texts["body"].str.count(pattern).sum()
    total_match_summary.append((name, total_matches))

total_match_summary_df = pd.DataFrame(total_match_summary, columns=["pattern_name", "total_matches"])
total_match_summary_df = total_match_summary_df.sort_values("total_matches", ascending=False)

total_match_summary_df

,pattern_name,total_matches
0,markdown_heading,351
1,markdown_image,347
2,markdown_link,347
6,guillemets_quotes,280
8,dialogue_dash,131
7,curly_double_quotes,20
3,bullet_list,11
5,html_tag,4
4,numbered_list,1


In [31]:
# Create one boolean column per pattern to indicate if it is present in each row
for name, pattern in pattern_checks.items():
    texts[f"has_{name}"] = texts["body"].str.contains(pattern, regex=True, na=False)

# Show a few of the new columns
pattern_feature_cols = [col for col in texts.columns if col.startswith("has_")]
texts[pattern_feature_cols].head()

,has_markdown_heading,has_markdown_image,has_markdown_link,has_bullet_list,has_numbered_list,has_html_tag,has_guillemets_quotes,has_curly_double_quotes,has_dialogue_dash
0,True,True,True,False,False,False,False,False,False
1,True,True,True,False,False,False,True,False,False
2,True,True,True,False,False,False,False,False,True
3,True,True,True,False,False,False,True,False,False
4,True,True,True,False,False,False,False,False,False


In [32]:
# Create count columns showing how many times each pattern appears in a row

for name, pattern in pattern_checks.items():
    texts[f"count_{name}"] = texts["body"].str.count(pattern)

count_feature_cols = [col for col in texts.columns if col.startswith("count_")]
texts[count_feature_cols].head()

,count_markdown_heading,count_markdown_image,count_markdown_link,count_bullet_list,count_numbered_list,count_html_tag,count_guillemets_quotes,count_curly_double_quotes,count_dialogue_dash
0,4,2,2,0,0,0,0,0,0
1,1,3,3,0,0,0,2,0,0
2,1,4,4,0,0,0,0,0,30
3,1,2,2,0,0,0,2,0,0
4,5,2,2,0,0,0,0,0,0


In [ ]:
# Rows with many special symbols often contain markup, or formatting

def special_symbol_count(text):
    return len(re.findall(r"[^\w\s+ª+++Ñ+å+ÿ+à]", text))

texts["special_symbol_count"] = texts["body"].apply(special_symbol_count)

texts.sort_values("special_symbol_count", ascending=False)[
    ["sanity_text_id", "title", "special_symbol_count", "body"]
].head(10)

,sanity_text_id,title,special_symbol_count,body
54,52b4157f-0a92-40fe-bf18-a4f9819b175d,Da h++nsene skulle til fotograf,164,## Da h++nsene skulle til fotograf\r\n\r\nEn da...
79,88100afd-75df-47fb-bd79-ba6dbbaa895d,Da endene fikk eget spa i hagen,154,## Da endene fikk eget spa i hagen\r\n\r\n![3 ...
2,02df17fa-3bfa-4f87-8cd5-d523c12c9a1f,Snork,153,## Snork\r\n\r\nGÇô Jeg vil ikke sove p+Ñ rommet ...
35,2cc2fc13-38d7-4184-a301-88ba0f433bd4,Bestemor,153,## Bestemor\r\n\r\n![Eldre dame sitter i gynge...
6,06a147ad-07a9-4bf5-a012-7f5f3b33a0c1,Rasmus er ulydig,150,## Rasmus er ulydig\r\n\r\n![Spraglete katt]()...
20,1817ea02-414e-4110-8cdf-a6816c4c817a,Tim Burton,142,## Tim Burton\r\n\r\n### Tim Burton\r\n\r\n25....
90,9cec37e6-8377-4bb7-b7a8-c342fbc40cdd,Sjokolademaskinen,137,## Sjokolademaskinen\r\n\r\nTenk deg at dette ...
18,14bdcf5a-e89b-4217-9f81-ba6cb4b39d06,Sn++mannen,133,"## Sn++mannen\r\n\r\nDet sn++r. Store, hvite fla..."
145,f37a1282-7ebe-4818-a7da-6ad5fcd9c08a,Fiona drar til tannlegen,133,## Fiona drar til tannlegen\r\n\r\n![Svart og ...
105,b4435de9-474f-44da-98f6-71a8d2d9509f,"Lange, raske tog",127,"## Lange, raske tog\r\n\r\nLaffen og Henrik g+Ñ..."


In [34]:
# This helper function returns a few example rows containing a selected pattern

def show_pattern_examples(df, pattern_name, text_col="body", n=5):
    pattern = pattern_checks[pattern_name]
    
    result = df[df[text_col].str.contains(pattern, regex=True, na=False)][
        ["sanity_text_id", "serialNumber", "title", text_col]
    ].head(n)
    
    return result

In [35]:
# Show a few rows containing Markdown heading syntax
show_pattern_examples(texts, "markdown_heading", n=5)

,sanity_text_id,serialNumber,title,body
0,01ff8ad2-ef5b-4cf2-8f1d-b44663790fa4,HT_105_+ÿrner,+ÿrner,## +ÿrner\r\n\r\n![+ÿrn med fisk i kl++rne. Foto....
1,0202b907-4516-4beb-8e07-759e473b5ea2,HT_43_Astronaut,Astronaut,## Astronaut\r\n\r\n![Astronaut viser tommel o...
2,02df17fa-3bfa-4f87-8cd5-d523c12c9a1f,HT_04_Snork,Snork,## Snork\r\n\r\nGÇô Jeg vil ikke sove p+Ñ rommet ...
3,035af603-c895-4f40-a3e8-4b05a07a09bd,HT_125_Miniatyrer-William,Miniatyrer - gjennom n+Ñl++yet,## Miniatyrer - gjennom n+Ñl++yet\r\n\r\n![Forsi...
4,05c18c0e-b6b5-40b8-9009-130b044693c6,HT_135_NRK,NRK,## NRK\r\n\r\n![NRK-bygget Marienlyst i Oslo](...


In [36]:
# Show a few rows containing Markdown image syntax
show_pattern_examples(texts, "markdown_image", n=5)

,sanity_text_id,serialNumber,title,body
0,01ff8ad2-ef5b-4cf2-8f1d-b44663790fa4,HT_105_+ÿrner,+ÿrner,## +ÿrner\r\n\r\n![+ÿrn med fisk i kl++rne. Foto....
1,0202b907-4516-4beb-8e07-759e473b5ea2,HT_43_Astronaut,Astronaut,## Astronaut\r\n\r\n![Astronaut viser tommel o...
2,02df17fa-3bfa-4f87-8cd5-d523c12c9a1f,HT_04_Snork,Snork,## Snork\r\n\r\nGÇô Jeg vil ikke sove p+Ñ rommet ...
3,035af603-c895-4f40-a3e8-4b05a07a09bd,HT_125_Miniatyrer-William,Miniatyrer - gjennom n+Ñl++yet,## Miniatyrer - gjennom n+Ñl++yet\r\n\r\n![Forsi...
4,05c18c0e-b6b5-40b8-9009-130b044693c6,HT_135_NRK,NRK,## NRK\r\n\r\n![NRK-bygget Marienlyst i Oslo](...


In [39]:
# Show a few rows containing Norwegian/French-style quotation marks -½ -+
show_pattern_examples(texts, "guillemets_quotes", n=5)

,sanity_text_id,serialNumber,title,body
1,0202b907-4516-4beb-8e07-759e473b5ea2,HT_43_Astronaut,Astronaut,## Astronaut\r\n\r\n![Astronaut viser tommel o...
3,035af603-c895-4f40-a3e8-4b05a07a09bd,HT_125_Miniatyrer-William,Miniatyrer - gjennom n+Ñl++yet,## Miniatyrer - gjennom n+Ñl++yet\r\n\r\n![Forsi...
6,06a147ad-07a9-4bf5-a012-7f5f3b33a0c1,HT_47_Rasmus,Rasmus er ulydig,## Rasmus er ulydig\r\n\r\n![Spraglete katt]()...
7,0af832ca-a33b-4606-96ce-1904d99101d3,HT_143_Unionsoppl++sningen,Unionsoppl++sningen,## Unionsoppl++sningen\r\n\r\n![Det norske og d...
8,0b8506ca-81ea-4c3f-804f-2bdfa9d1f359,HT_29_Jaktkatt,Hvordan kan katter jakte om natten?,## Hvordan kan katter jakte om natten?\r\n\r\n...


In [42]:
# This function extracts the matched strings themselves from a text column
# Useful for seeing the exact recurring expressions

def extract_matches(text, pattern):
    return re.findall(pattern, text)

# Example: extract all Markdown image-like strings
texts["matches_markdown_image"] = texts["body"].apply(
    lambda x: extract_matches(x, pattern_checks["markdown_image"])
)

texts[texts["matches_markdown_image"].apply(len) > 0][
    ["sanity_text_id", "title", "matches_markdown_image"]
].head(10)

,sanity_text_id,title,matches_markdown_image
0,01ff8ad2-ef5b-4cf2-8f1d-b44663790fa4,+ÿrner,"[![+ÿrn med fisk i kl++rne. Foto.](), ![Hvithode..."
1,0202b907-4516-4beb-8e07-759e473b5ea2,Astronaut,[![Astronaut viser tommel opp mens han sitter ...
2,02df17fa-3bfa-4f87-8cd5-d523c12c9a1f,Snork,[![Jente kjefter p+Ñ gutt; en hval i en snakkeb...
3,035af603-c895-4f40-a3e8-4b05a07a09bd,Miniatyrer - gjennom n+Ñl++yet,[![Forsiden av Willard Wigans Instagramside]()...
4,05c18c0e-b6b5-40b8-9009-130b044693c6,NRK,"[![NRK-bygget Marienlyst i Oslo](), ![Svart/hv..."
5,068108d2-6d62-4702-8037-f309323f7cd8,Elg,"[![Elg i tett bj++rkeskog. Foto.](), ![Elgfamil..."
6,06a147ad-07a9-4bf5-a012-7f5f3b33a0c1,Rasmus er ulydig,"[![Spraglete katt](), ![spraglete katt]()]"
7,0af832ca-a33b-4606-96ce-1904d99101d3,Unionsoppl++sningen,[![Det norske og det svenske flagget ligger sk...
8,0b8506ca-81ea-4c3f-804f-2bdfa9d1f359,Hvordan kan katter jakte om natten?,[![Gr+Ñ katt som l++per p+Ñ gr++nn plen i m++rket. ...
9,0d9a3616-148c-4636-b310-c81355014aae,Golden Gate Bridge,"[![Golden Gate Bridge - lang, r++d bro over bl+Ñ..."


In [43]:
# This cell finds repeated full lines across the dataset.
# Repeated lines can reveal boilerplate, headings, repeated captions, or template text.

all_lines = []

for text in texts["body"]:
    for line in text.splitlines():
        line = line.strip()
        if line:
            all_lines.append(line)

line_counts = Counter(all_lines)

repeated_lines_df = pd.DataFrame(
    [(line, count) for line, count in line_counts.items() if count >= 2],
    columns=["line", "count"]
).sort_values("count", ascending=False)

repeated_lines_df.head(50)

,line,count
6,###,16
2,### Beskrivelse,14
0,### Levevis,14
3,### Formering,11
1,### Historie,8
9,"Fra Nemi-bladet, omarbeidet",8
8,### Truet art,5
52,## Rare nytt+Ñrstradisjoner,5
5,Omarbeidet fra Nemi-bladet,4
65,![Jente gamer](),4


In [44]:
# This cell finds symbolic prefixes at the start of lines.
# For example: ##, ![, -, *, 1., etc.

def symbolic_prefix(line):
    match = re.match(r"^[^\w+ª+++Ñ+å+ÿ+à\s]+", line.strip())
    return match.group(0) if match else None

prefixes = []

for text in texts["body"]:
    for line in text.splitlines():
        line = line.strip()
        if line:
            prefix = symbolic_prefix(line)
            if prefix:
                prefixes.append(prefix)

prefix_counts = Counter(prefixes)

prefix_df = pd.DataFrame(prefix_counts.items(), columns=["prefix", "count"]).sort_values("count", ascending=False)
prefix_df.head(30)

,prefix,count
1,![,347
2,###,192
0,##,159
3,GÇô,125
4,-½,32
7,.,12
8,-,11
10,(,9
5,GÇ£,6
9,",",4


In [45]:
# This gives a small summary focused on Norwegian-specific cues

norwegian_audit = {
    "rows_with_guillemets_quotes": texts["has_guillemets_quotes"].sum(),
    "rows_with_dialogue_dash": texts["has_dialogue_dash"].sum(),
    "rows_with_curly_double_quotes": texts["has_curly_double_quotes"].sum(),
}

pd.DataFrame(norwegian_audit.items(), columns=["metric", "value"])

,metric,value
0,rows_with_guillemets_quotes,46
1,rows_with_dialogue_dash,13
2,rows_with_curly_double_quotes,4


In [46]:
# Save the enriched dataset with pattern flags and counts
texts.to_csv("../data/processed/question_texts_with_pattern_flags.csv", index=False)

print("Saved pattern-discovery output.")

Saved pattern-discovery output.
